# 09 · Speculative sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/09_speculative_sampling.ipynb) &nbsp; 📖 Concept deck: [Speculative decoding](https://nilesh-salpe.github.io/visual-ai/inference/06-speculative-decoding/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** draft-and-verify decoding with the modified rejection rule (Leviathan et al. 2023; Chen et al. 2023), proven by a Monte Carlo test to sample from exactly the target distribution, plus the expected-speedup formula.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


V = 6
def _norm(p): p = np.asarray(p, float); return p / p.sum()
_T = np.array([_norm(np.random.default_rng(i).random(V) ** 2 + .05) for i in range(V)])          # target: next-token table by last token
_D = np.array([_norm(0.6 * _T[i] + 0.4 * np.random.default_rng(100 + i).random(V)) for i in range(V)])  # draft: a noisy copy

def target_probs(prefix): return _T[prefix[-1]]
def draft_probs(prefix):  return _D[prefix[-1]]
print('toy models ready: vocab', V, '| target_probs(prefix), draft_probs(prefix)')

## Task 1: One draft-and-verify step

1. **Draft** γ tokens autoregressively from `draft_probs`, remembering each draft distribution $q_i$.
2. **Score** with the target: $p_i$ = `target_probs(prefix + drafts[:i])` for $i = 0 … γ$ (γ+1 distributions; on a GPU this is **one** batched forward pass).
3. For each draft $x_i$: **accept** with probability $\min(1, p_i(x_i)/q_i(x_i))$. On the first rejection, sample a replacement from $\mathrm{norm}(\max(0, p_i - q_i))$ and stop.
4. If all γ are accepted, also sample a **bonus** token from $p_γ$.

Return `(new_tokens, n_accepted)`. Use `rng.random()` and `rng.choice(V, p=…)`.

In [ ]:
def speculative_step(prefix, target_fn, draft_fn, gamma, rng):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_speculative_step(f):
    rng = np.random.default_rng(0); N = 30000; prefix = [2]
    first, pairs = np.zeros(V), np.zeros((V, V))
    for _ in range(N):
        toks, acc = f(prefix, target_probs, draft_probs, 3, rng)
        assert 1 <= len(toks) <= 4 and len(toks) == acc + 1, "must return accepted drafts + exactly one extra token"
        if len(toks) < 2:
            toks = toks + f(prefix + toks, target_probs, draft_probs, 3, rng)[0]
        first[toks[0]] += 1; pairs[toks[0], toks[1]] += 1
    assert np.abs(first / N - _T[2]).max() < 0.015, "first-token distribution does not match the TARGET model"
    joint = _T[2][:, None] * _T
    assert np.abs(pairs / N - joint).max() < 0.012, "two-token joint distribution does not match the target"

check("speculative_step", _test_speculative_step, speculative_step)

## Task 2: Expected tokens per target pass

With acceptance rate α (Leviathan et al. eq. 1), each target forward pass yields on average
$$E[\#\text{tokens}] = \frac{1-\alpha^{\gamma+1}}{1-\alpha}$$
For a fixed draft distribution $q$ and target distribution $p$, $\alpha = \sum_x \min(p(x), q(x))$.
Implement `expected_tokens(alpha, gamma)`. The tests also check it against your sampler.

In [ ]:
def expected_tokens(alpha, gamma):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_expected_tokens(f):
    assert abs(f(0.8, 4) - (1 - 0.8 ** 5) / 0.2) < 1e-9
    assert abs(f(0.0, 5) - 1.0) < 1e-9, "alpha = 0 -> 1 token per pass"
    p, q = _T[0], _D[0]; a = np.minimum(p, q).sum()
    rng = np.random.default_rng(1); n = 20000; tot = 0
    for _ in range(n):                       # simulate accept/reject only: draft x ~ q, accept w.p. min(1, p/q)
        k = 0
        while k < 4:
            x = rng.choice(V, p=q)
            if rng.random() < min(1, p[x] / q[x]): k += 1
            else: break
        tot += k + 1
    mean = tot / n
    assert abs(mean - f(a, 4)) / f(a, 4) < 0.03, f"simulated {mean:.3f} tokens/pass vs formula {f(a, 4):.3f}"

check("expected_tokens", _test_expected_tokens, expected_tokens)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: One draft-and-verify step
def speculative_step_solution(prefix, target_fn, draft_fn, gamma, rng):
    ctx, drafts, qs = list(prefix), [], []
    for _ in range(gamma):                                        # 1. draft
        q = draft_fn(ctx); x = int(rng.choice(len(q), p=q))
        drafts.append(x); qs.append(q); ctx.append(x)
    ps = [target_fn(list(prefix) + drafts[:i]) for i in range(gamma + 1)]   # 2. verify (one batched pass)
    out = []
    for i, x in enumerate(drafts):                                # 3. accept / reject
        if rng.random() < min(1.0, ps[i][x] / qs[i][x]):
            out.append(x)
        else:
            resid = np.maximum(ps[i] - qs[i], 0); resid /= resid.sum()
            out.append(int(rng.choice(len(resid), p=resid)))
            return out, i
    out.append(int(rng.choice(len(ps[gamma]), p=ps[gamma])))      # 4. bonus token
    return out, gamma

check("speculative_step (reference)", _test_speculative_step, speculative_step_solution)

In [ ]:
# @title Solution for Task 2: Expected tokens per target pass
def expected_tokens_solution(alpha, gamma):
    return (1 - alpha ** (gamma + 1)) / (1 - alpha) if alpha < 1 else gamma + 1

check("expected_tokens (reference)", _test_expected_tokens, expected_tokens_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Wall-clock speedup (Leviathan et al. Theorem 3.8): E[tokens] / (gamma*c + 1), where c = draft cost / target cost
for alpha in (0.6, 0.8, 0.9):
    best = max(range(1, 16), key=lambda g: expected_tokens_solution(alpha, g) / (g * 0.05 + 1))
    print(f"alpha={alpha}: best gamma={best:2d}  ->  speedup {expected_tokens_solution(alpha, best) / (best * 0.05 + 1):.2f}x  (c = 0.05)")

---
## Takeaways
- Decode is memory-bound. Verifying γ drafted tokens costs about the same as one target step, so accepted drafts are nearly free tokens.
- The modified rejection rule makes the output distribution **identical** to sampling from the target model. Your Monte Carlo test proves it on a toy model.
- Speedup depends on the acceptance rate α and the draft cost c. The papers reported **2–3×** (T5-XXL) and **2–2.5×** (Chinchilla 70B).